# SCoPE — materialize per-case extraction rows on existing cells

The first study commits printed the extraction table but did not persist per-case rows; the
paired arm-vs-arm test needs them. Games are deterministic (greedy decode, fixed seed), so
re-running extraction on the same cells reproduces the same outcomes — this notebook does
exactly that with the current scripts, which write `extraction.jsonl` (and optionally
`validation.jsonl`) into each cell.

**Before running:** Settings → Accelerator **GPU T4 x2** · Internet **on** · **Add Input** →
attach the dataset holding your downloaded `scope_<dataset>_<tag>_seed<seed>.zip` files.
Rough budget: ~1–2 h per 7B cell at limit 40; the default (AND cells only) fits one commit easily.

In [ ]:
import os, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working')
os.chdir(WORK)

def fetch(name, url):
    if (WORK / name).exists():
        subprocess.run(['git', '-C', name, 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', url, name], check=True)

fetch('lineup', 'https://github.com/santoshcheethiralame-dot/LINEUP')
fetch('scope', 'https://github.com/santoshcheethiralame-dot/SCoPE')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', './lineup[gpu]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', './scope'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'bitsandbytes>=0.46.1'], check=True)
import bitsandbytes
print('[ok] lineup + scope installed; bitsandbytes', bitsandbytes.__version__)

In [ ]:
CONDITIONS = ('and',)     # add 'hardtraps' for the OR cells too (roughly doubles the runtime)
EXTRACT_LIMIT = 40        # wrong cases per cell; a superset of the original run's cases
N_SAMPLES = 48
SEED = 0
VALIDATE = False          # True also re-runs validate_designed for per-case validation.jsonl rows
VALIDATE_MAX_SIZE = 3

MODEL_ZOO = {
    'qwen':    'Qwen/Qwen2.5-7B-Instruct',
    'phi':     'microsoft/Phi-3.5-mini-instruct',
    'mistral': 'mistralai/Mistral-7B-Instruct-v0.3',
}

In [ ]:
# Unzip every attached scope results zip into runs/ (they were archived from a runs/ root).
import zipfile

zips = sorted(Path('/kaggle/input').glob('**/scope_*.zip'))
assert zips, 'no scope_*.zip found under /kaggle/input -- attach the results dataset'
for path in zips:
    with zipfile.ZipFile(path) as archive:
        archive.extractall('runs')
    print('unzipped', path.name)

cells = sorted(
    cell for cond in CONDITIONS for cell in Path('runs').glob(f'*/{cond}/*')
    if (cell / 'scenarios.jsonl').exists() and cell.name in MODEL_ZOO
)
assert cells, 'no matching cells -- check CONDITIONS and the zip contents'
print('cells to process:', [str(c) for c in cells])

In [ ]:
for cell in cells:
    model = MODEL_ZOO[cell.name]
    print(f'==== extraction: {cell}  ({model}) ====', flush=True)
    subprocess.run(
        [sys.executable, 'scope/scripts/run_extraction.py', '--cell', str(cell),
         '--model', model, '--load-in-4bit', '--n-samples', str(N_SAMPLES),
         '--seed', str(SEED), '--limit', str(EXTRACT_LIMIT)],
        check=True,
    )
    assert (cell / 'extraction.jsonl').stat().st_size > 0
    if VALIDATE:
        print(f'==== validation: {cell} ====', flush=True)
        subprocess.run(
            [sys.executable, 'scope/scripts/validate_designed.py', '--cell', str(cell),
             '--model', model, '--load-in-4bit', '--max-size', str(VALIDATE_MAX_SIZE)],
            check=True,
        )
        assert (cell / 'validation.jsonl').stat().st_size > 0
    print(f'[ok] {cell}', flush=True)

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/scope_extraction_rows', 'zip', 'runs')
print('download scope_extraction_rows.zip from the Output panel')